In [ ]:
%%bash
### Extract unique sequences (by hash) from ATB + CF viruses to create UHVDB r2025_10 ###

# # updated checkv database here: /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/complete_genomes_r2025_10/combined/combined_complete_genomes/checkv/update/checkv_db_20250908

# # combine atb hq viruses
# cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/atb/r2025_10/atb_batch_*/uhvdb/hqfilter/atb_batch_*.hq_viruses.fna.gz \
#     > combined_atb_hq_viruses.fna.gz
# # count = 327,545

# # combine assembled cf metagenome viruses
# cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/prj*/*/*_cf/uhvdb/hqfilter/*_cf.hq_viruses.fna.gz \
#     > combined_cf_metagenome_viruses.fna.gz
# # count = 4,734

# # combine all new cf viruses
# cat combined_atb_hq_viruses.fna.gz combined_cf_metagenome_viruses.fna.gz \
#     > combined_hq_cf_viruses.fasta.gz
# # count = 332,279

# # trim DTRs on new viruses
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/tr-trimmer \
#     combined_hq_cf_viruses.fasta.gz \
#     --min-length 20 --include-tr-info \
#     > combined_hq_cf_viruses.tr-trimmer.fna

# # calculate hash of new sequences
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/seq-hasher \
#     combined_hq_cf_viruses.tr-trimmer.fna \
#     --multi-kmer-hashing \
#     --circular-kmers \
#     > combined_hq_cf_viruses.seq-hasher.tsv

# # combine UHVDB and CF seq-hasher outputs to identify unique hashes
# cat ../uhvdb_final_files/r2025_09/uhvdb_seqhashes.tsv.gz \
#     combined_hq_cf_viruses.seq-hasher.tsv.gz \
#     > combined.seq-hasher.tsv.gz

# # dereplicate hashes
# csvtk uniq \
#     combined.seq-hasher.tsv.gz \
#     --no-header-row \
#     --fields 2 \
#     --tabs \
#     --out-file combined_hq_cf_viruses.csvtk_uniq.tsv.gz

# # extract seq ids
# csvtk cut \
#     --tabs \
#     --fields 1 \
#     combined_hq_cf_viruses.csvtk_uniq.tsv.gz \
#     --out-file combined_hq_cf_viruses.unique_seq_ids.tsv

# # extract unique sequences
# seqkit \
#     grep \
#     --pattern-file combined_hq_cf_viruses.unique_seq_ids.tsv \
#     combined_hq_cf_viruses.tr-trimmer.fna \
#     -o combined_hq_cf_viruses.unique.fna.gz

# # combine with previous UHVDB unique sequences
# cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_unique.fna.gz \
#     combined_hq_cf_viruses.unique.fna.gz \
#     > /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_unique.fna.gz

In [ ]:
%%bash
### Dereplicate new unique sequences at 100% ANI and 100% AF (shorter sequence) ###

# # prefilter with 99% identity (so this can also be used for genomovars)
# vclust \
#     prefilter \
#     --in combined_hq_cf_viruses.unique.fna.gz \
#     --out combined_hq_cf_viruses.vclust_dedup_prefilter.txt \
#     --threads 32 \
#     --min-ident 0.99

# # align with 99.5% identity (so this can also be used for genomovars)
# vclust \
#     align \
#     --in combined_hq_cf_viruses.unique.fna.gz \
#     --out combined_hq_cf_viruses.vclust_ani995_qcov100_ani.tsv \
#     --filter combined_hq_cf_viruses.vclust_dedup_prefilter.txt \
#     --filter-threshold 0.99 \
#     --threads 32 \
#     --out-ani 0.995 \
#     --out-qcov 1.0

# # cluster with 100% identity and 100% coverage (shorter sequence)
# vclust \
#     cluster \
#     --in combined_hq_cf_viruses.vclust_ani995_qcov100_ani.tsv \
#     --ids combined_hq_cf_viruses.vclust_ani995_qcov100_ani.ids.tsv \
#     --out combined_hq_cf_viruses.vclust_ani100_qcov100_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 1.0 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     combined_hq_cf_viruses.vclust_ani100_qcov100_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file combined_hq_cf_viruses.vclust_ani100_qcov100_reps.tsv

# seqkit \
#     grep \
#     combined_hq_cf_viruses.unique.fna.gz \
#     --pattern-file combined_hq_cf_viruses.vclust_ani100_qcov100_reps.tsv \
#     --out-file combined_hq_cf_viruses.vclust_ani100_qcov100_reps.fna.gz

In [ ]:
%%bash
### align new dereplicated sequences to dereplicated UHVDB and dereplicate all at 100% ANI and 100% AF (shorter sequence) ###

# # align new dereplicated sequences to UHVDB
# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_dedup_reps.fna.gz" > uhvd_dedup_path.txt

# # build kmer-db database for UHVDB dedup sequences
# kmer-db \
# 	build \
# 	-k 25 \
# 	-f 0.2 \
# 	-t 32 \
# 	-multisample-fasta \
# 	uhvd_dedup_path.txt \
# 	uhvd_dedup.kdb

# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_viruses.vclust_ani100_qcov100_reps.fna.gz" > cf_dedup_path.txt

# # compare deduplicated CF sequences to UHVDB dedup sequences
# kmer-db \
# 	new2all \
# 	-sparse \
# 	-min num-kmers:20 \
# 	-min ani-shorter:1.0 \
# 	-t 32 \
# 	-multisample-fasta \
# 	uhvd_dedup.kdb \
# 	cf_dedup_path.txt \
# 	cf_v_uhvdb.csv

# # convert kmer-db output to LZ-ANI filter format
# kmer-db \
# 	distance \
# 	ani-shorter \
# 	-sparse \
# 	-min 1.0 \
# 	-t 32 \
# 	cf_v_uhvdb.csv \
# 	cf_v_uhvdb.dist.csv

# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/kmerdb_new2all_to_lzani.py \
# 	-i cf_v_uhvdb.dist.csv \
# 	-o cf_v_uhvdb.vclust_prefilter.txt

# zcat ../uhvdb_final_files/r2025_09/uhvdb_dedup_reps.fna.gz > cf_v_uhvdb.ref_query_combined.fna
# zcat combined_hq_cf_viruses.vclust_ani100_qcov100_reps.fna.gz >> cf_v_uhvdb.ref_query_combined.fna

# # align with 100% identity and 100% coverage (shorter sequence)
# vclust \
#     align \
#     --in cf_v_uhvdb.ref_query_combined.fna \
#     --out cf_v_uhvdb.vclust_ani100_qcov100_ani.tsv \
#     --filter cf_v_uhvdb.vclust_prefilter.txt \
#     --filter-threshold 1.0 \
#     --threads 32 \
#     --out-ani 1.0 \
#     --out-qcov 1.0

# vclust \
#     cluster \
#     --in cf_v_uhvdb.vclust_ani100_qcov100_ani.tsv \
#     --ids cf_v_uhvdb.vclust_ani100_qcov100_ani.ids.tsv \
#     --out cf_v_uhvdb.vclust_ani100_qcov100_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 1.0 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     cf_v_uhvdb.vclust_ani100_qcov100_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file cf_v_uhvdb.vclust_ani100_qcov100_reps.tsv

# seqkit \
#     grep \
#     cf_v_uhvdb.ref_query_combined.fna \
#     --pattern-file cf_v_uhvdb.vclust_ani100_qcov100_reps.tsv \
#     --out-file /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_dedup_reps.fna.gz

In [ ]:
%%bash
### cluster new sequences at genomovar level 99.5% ANI and 100% AF (shorter sequence) ###

# vclust \
#     align \
#     --in combined_hq_cf_viruses.unique.fna.gz \
#     --out combined_hq_cf_viruses.vclust_ani995_qcov100_ani.tsv \
#     --filter combined_hq_cf_viruses.vclust_dedup_prefilter.txt \
#     --filter-threshold 0.99 \
#     --threads 32 \
#     --out-ani 0.995 \
#     --out-qcov 1.0

# vclust \
#     cluster \
#     --in combined_hq_cf_viruses.vclust_ani995_qcov100_ani.tsv \
#     --ids combined_hq_cf_viruses.vclust_ani995_qcov100_ani.ids.tsv \
#     --out combined_hq_cf_viruses.vclust_ani995_qcov100_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 0.995 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     combined_hq_cf_viruses.vclust_ani995_qcov100_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file combined_hq_cf_viruses.vclust_ani995_qcov100_reps.tsv

# seqkit \
#     grep \
#     combined_hq_cf_viruses.unique.fna.gz \
#     --pattern-file combined_hq_cf_viruses.vclust_ani995_qcov100_reps.tsv \
#     --out-file combined_hq_cf_viruses.vclust_ani995_qcov100_reps.fna.gz

### 65,470 genomovars in new sequences

In [ ]:
%%bash
### align new genomovars to UHVDB genomovars and cluster at genomovar level 99.5% ANI and 100% AF (shorter sequence) ###

# # align dereplicated sequences to UHVDB
# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_genomovars_reps.fna.gz" > uhvdb_genomovars_reps_path.txt

# # build kmer-db database for reference
# kmer-db \
# 	build \
# 	-k 25 \
# 	-f 0.2 \
# 	-t 32 \
# 	-multisample-fasta \
# 	uhvdb_genomovars_reps_path.txt \
# 	uhvdb_genomovars_reps.kdb

# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_viruses.vclust_ani995_qcov100_reps.fna.gz" > cf_genomovars_path.txt

# # compare query to reference
# kmer-db \
# 	new2all \
# 	-sparse \
# 	-min num-kmers:20 \
# 	-min ani-shorter:0.99 \
# 	-t 32 \
# 	-multisample-fasta \
# 	uhvdb_genomovars_reps.kdb \
# 	cf_genomovars_path.txt \
# 	cf_v_uhvdb_genomovars.csv

# # convert kmer-db output to LZ-ANI filter format
# kmer-db \
# 	distance \
# 	ani-shorter \
# 	-sparse \
# 	-min 0.99 \
# 	-t 32 \
# 	cf_v_uhvdb_genomovars.csv \
# 	cf_v_uhvdb_genomovars.dist.csv

# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/kmerdb_new2all_to_lzani.py \
# 	-i cf_v_uhvdb_genomovars.dist.csv \
# 	-o cf_v_uhvdb_genomovars.vclust_ani99_prefilter.txt

# zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_genomovars_full.fna.gz > cf_v_uhvdb_genomovars.ref_query_combined.fna
# zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_viruses.vclust_ani995_qcov100_reps.fna.gz >> cf_v_uhvdb_genomovars.ref_query_combined.fna

# vclust \
#     align \
#     --in cf_v_uhvdb_genomovars.ref_query_combined.fna \
#     --out cf_v_uhvdb_genomovars.vclust_ani995_qcov100_ani.tsv \
#     --filter cf_v_uhvdb_genomovars.vclust_ani99_prefilter.txt \
#     --filter-threshold 0.99 \
#     --threads 32 \
#     --out-ani 0.99 \
#     --out-qcov 1.0

# vclust \
#     cluster \
#     --in cf_v_uhvdb_genomovars.vclust_ani995_qcov100_ani.tsv \
#     --ids cf_v_uhvdb_genomovars.vclust_ani995_qcov100_ani.ids.tsv \
#     --out cf_v_uhvdb_genomovars.vclust_ani995_qcov100_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 0.995 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     cf_v_uhvdb_genomovars.vclust_ani995_qcov100_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file cf_v_uhvdb_genomovars.vclust_ani995_qcov100_reps.tsv

seqkit \
    grep \
    cf_v_uhvdb_genomovars.ref_query_combined.fna \
    --pattern-file cf_v_uhvdb_genomovars.vclust_ani995_qcov100_reps.tsv \
    --out-file /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_genomovars_reps_full.fna.gz

In [ ]:
### split new genomovars into confident and uncertain ###

# # combine r2025_10 virus classification metadata

# # combine all new mine reports into one file
# !for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/atb/r2025_10/*/uhvdb/minereport/*.mine_summary.tsv.gz; do
#     zcat "$file" | tail -n +2 >> cf_viruses.csvtk_concat.tsv
# done

# !for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/prj*/*/*/uhvdb/minereport/*.mine_summary.tsv.gz; do
#     zcat "$file" | tail -n +2 >> cf_viruses.csvtk_concat.tsv
# done

# # identify all new fasta IDs
# !zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/atb/r2025_10/*/uhvdb/virusfilter/*.uhvdb_viruses.fna.gz \
#     | grep "^>" > cf_virus_ids.txt

# !zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/prj*/*/*/uhvdb/virusfilter/*.uhvdb_viruses.fna.gz \
#     | grep "^>" >> cf_virus_ids.txt

# # extract r2025_10 HQ confident
import polars as pl

genomovar_reps = pl.read_csv('cf_v_uhvdb_genomovars.vclust_ani995_qcov100_reps.tsv', separator='\t')

r2025_10_meta = pl.read_csv('cf_viruses.csvtk_concat.tsv', separator='\t', has_header=False)

print(
    r2025_10_meta
        .filter(
            (pl.col('column_44').is_in(['confident', 'uncertain'])) &
            (pl.col('column_45').is_not_null()) &
            (pl.col('column_45') != 'NA')
        )
        .with_columns([
            pl.col('column_45').cast(pl.Float32).alias('completeness')
        ])
        .filter(
            (pl.col('completeness') >= 90.0) &
            (
                (~pl.col('column_48').str.contains('>1.5x')) |
                (pl.col('column_48').is_null())
            ) &
            (pl.col('column_1').is_in(set(genomovar_reps['cluster'])))
        )
        .group_by('column_44')
        .len()
)
# 58698 confident
# 5401 uncertain
(
    r2025_10_meta
        .filter(
            (pl.col('column_44').is_in(['uncertain'])) &
            (pl.col('column_45').is_not_null()) &
            (pl.col('column_45') != 'NA')
        )
        .with_columns([
            pl.col('column_45').cast(pl.Float32).alias('completeness')
        ])
        .filter(
            (pl.col('completeness') >= 90.0) &
            (
                (~pl.col('column_48').str.contains('>1.5x')) |
                (pl.col('column_48').is_null())
            ) &
            (pl.col('column_1').is_in(set(genomovar_reps['cluster'])))
        )
        [['column_1']]
        .write_csv('cf_v_uhvdb_genomovars.uncertain.tsv', separator='\t', include_header=True)
)


shape: (2, 2)
┌───────────┬───────┐
│ column_44 ┆ len   │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ uncertain ┆ 5401  │
│ confident ┆ 58698 │
└───────────┴───────┘


In [16]:
# # extract r2025_10 HQ uncertain genomes that are in genomovar representatives
# !seqkit grep \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_genomovars_reps_full.fna.gz \
#     --pattern-file cf_v_uhvdb_genomovars.uncertain.tsv \
#     --out-file cf_genomovars.uncertain.fna.gz

# !seqkit split2 \
#     cf_genomovars.uncertain.fna.gz \
#     --by-size 100 \
#     --out-dir cf_genomovars_uncertain_split

# # run hmmsearch on uncertain genomes to classify further
# sbatch hallmark_hmmsearch.sh

import polars as pl
import glob

# count hmm hallmark hits per genome
genomad_virus_hallmarks = set(
    pl.read_csv('../figure_s5/genomad_metadata_v1.9.tsv.gz', separator='\t', ignore_errors=True)
    .filter(
        (pl.col('VIRUS_HALLMARK') == 1)
    )['MARKER']
)

genomad_plasmid_hallmarks = set(
    pl.read_csv('../figure_s5/genomad_metadata_v1.9.tsv.gz', separator='\t', ignore_errors=True)
    .filter(
        (pl.col('PLASMID_HALLMARK') == 1)
    )['MARKER']
)

results = []
for file in glob.glob('hmm_results/*.tbl'):
    with open(file, 'r') as tbl:
        for line in tbl:
            if '#' in line[0]:
                continue
            strip_split = line.strip().split()
            protein = strip_split[0]
            genome = protein.rsplit('_', 1)[0]
            target = strip_split[2]
            results.append({'genome': genome, 'protein': protein, 'hallmark': target, 'evalue': float(strip_split[4])})
    tbl.close()

pl.DataFrame(results).unique('genome')

uncertain_hmm_hallmarks = (
    pl.DataFrame(results)
        .with_columns([
            pl.when(pl.col('hallmark').is_in(genomad_virus_hallmarks)).then(1).otherwise(0).alias('virus_hallmarks'),
            pl.when(pl.col('hallmark').is_in(genomad_plasmid_hallmarks)).then(1).otherwise(0).alias('plasmid_hallmarks'),
        ])
        .sort('evalue', descending=False)
        .group_by('protein')
        .first()
        .group_by(['genome'])
        .agg([pl.col('virus_hallmarks').sum().alias('virus_hallmarks'), pl.col('plasmid_hallmarks').sum().alias('plasmid_hallmarks')])
)

# write out genomes with >= 3 virus hallmarks and 0 plasmid hallmarks (would add 1 point to score)
uncertain2confident = (
    uncertain_hmm_hallmarks
        .filter(
            (pl.col('virus_hallmarks') >= 3) &
            (pl.col('plasmid_hallmarks') == 0)
        )
)

uncertain2confident[['genome']].write_csv('cf_genomovars_uncertain2confident.txt', include_header=False)

In [ ]:
# # extract uncertain to confident genomes
# !seqkit grep \
#     cf_genomovars.uncertain.fna.gz \
#     --pattern-file cf_genomovars_uncertain2confident.txt \
#     --out-file cf_genomovars_uncertain2confident.fna.gz

# # extract cf confident genomes
# !seqkit grep \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_viruses.vclust_ani995_qcov100_reps.fna.gz \
#     --pattern-file cf_v_uhvdb_genomovars.vclust_ani995_qcov100_reps.tsv | \
# seqkit grep \
#     --invert-match \
#     --pattern-file cf_v_uhvdb_genomovars.uncertain.tsv \
#     --out-file combined_hq_cf_viruses_confident.fna.gz

# !seqkit grep \
#     cf_genomovars.uncertain.fna.gz \
#     --invert-match \
#     --pattern-file cf_genomovars_uncertain2confident.txt \
#     --out-file cf_genomovars_still_uncertain.fna.gz

# # combine r2025_10 uncertain2confident genomes with r2025_09 confident + r2025_10 confident
# !cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_confident_genomovars_reps.fna.gz \
#     combined_hq_cf_viruses_confident.fna.gz \
#     cf_genomovars_uncertain2confident.fna.gz \
#     > /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_confident_genomovars_reps.fna.gz

In [18]:
r2025_10_meta = pl.read_csv('cf_viruses.csvtk_concat.tsv', separator='\t', has_header=False)
print(
    r2025_10_meta
        .filter(
            (pl.col('column_44').is_in(['confident', 'uncertain'])) &
            (pl.col('column_45').is_not_null()) &
            (pl.col('column_45') != 'NA')
        )
        .with_columns([
            pl.col('column_45').cast(pl.Float32).alias('completeness')
        ])
        .filter(
            (pl.col('completeness') >= 90.0) &
            (
                (~pl.col('column_48').str.contains('>1.5x')) |
                (pl.col('column_48').is_null())
            ) &
            (pl.col('column_1').is_in(set(genomovar_reps['cluster'])))
        )
        .group_by('column_44')
        .len()
)

shape: (2, 2)
┌───────────┬───────┐
│ column_44 ┆ len   │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ confident ┆ 58698 │
│ uncertain ┆ 5401  │
└───────────┴───────┘


In [ ]:
print("Number converted from uncertain to confident:",
    len(set(pl.read_csv('cf_genomovars_uncertain2confident.txt', has_header=False)['column_1']))
)

Number converted from uncertain to confident: 3339


In [ ]:
%%bash
# # combine r2025_10 uncertain2confident genomes with r2025_10 confident
# !cat combined_hq_cf_viruses_confident.fna.gz \
#     cf_genomovars_uncertain2confident.fna.gz \
#     > combined_hq_cf_genomovars_confident.fna.gz

# ### align dereplicates sequences v self to get clustering at 95% ANI and 85% AF ###
# vclust \
#     prefilter \
#     --in combined_hq_cf_genomovars_confident.fna.gz \
#     --out combined_hq_cf_genomovars_confident.vclust_votu_prefilter.txt \
#     --threads 32 \
#     --min-ident 0.95

# vclust \
#     align \
#     --in combined_hq_cf_genomovars_confident.fna.gz \
#     --out combined_hq_cf_genomovars_confident.vclust_votu_ani.tsv \
#     --filter combined_hq_cf_genomovars_confident.vclust_votu_prefilter.txt \
#     --threads 32 \
#     --out-ani 0.95 \
#     --out-qcov 0.85

# csvtk cut \
#     combined_hq_cf_genomovars_confident.vclust_votu_ani.tsv \
#     --tabs \
#     --out-tabs \
#     --delete-header \
#     --fields query,reference,gani \
#     --out-file combined_hq_cf_viruses.vclust_votu_gani.tsv.gz

In [ ]:
%%bash
# # align dereplicated sequences to UHVDB
# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_confident_genomovars_reps.fna.gz" > uhvdb_confident_genomovars_path.txt

# # build kmer-db database for reference
# kmer-db \
# 	build \
# 	-k 25 \
# 	-f 0.2 \
# 	-t 48 \
# 	-multisample-fasta \
# 	uhvdb_confident_genomovars_path.txt \
# 	uhvdb_confident_genomovars.kdb

# echo "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_genomovars_confident.fna.gz" > cf_confident_genomovar_path.txt

# # compare query to reference
# kmer-db \
# 	new2all \
# 	-sparse \
# 	-min num-kmers:20 \
# 	-min ani-shorter:0.95 \
# 	-t 48 \
# 	-multisample-fasta \
# 	uhvdb_confident_genomovars.kdb \
# 	cf_confident_genomovar_path.txt \
# 	cf_v_uhvdb_votu.csv

# # convert kmer-db output to LZ-ANI filter format
# kmer-db \
# 	distance \
# 	ani-shorter \
# 	-sparse \
# 	-min 0.95 \
# 	-t 48 \
# 	cf_v_uhvdb_votu.csv \
# 	cf_v_uhvdb_votu.dist.csv

# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/kmerdb_new2all_to_lzani.py \
# 	-i cf_v_uhvdb_votu.dist.csv \
# 	-o cf_v_uhvdb_votu.vclust_votu_prefilter.txt

# zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_confident_genomovars_reps.fna.gz > cf_v_uhvdb_votu.ref_query_combined.fna
# zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_cf_supplement/combined_hq_cf_genomovars_confident.fna.gz >> cf_v_uhvdb_votu.ref_query_combined.fna

# vclust \
#     align \
#     --in cf_v_uhvdb_votu.ref_query_combined.fna \
#     --out cf_v_uhvdb_votu.vclust_votu_ani.tsv \
#     --filter cf_v_uhvdb_votu.vclust_votu_prefilter.txt \
#     --filter-threshold 0.95 \
#     --threads 48 \
#     --out-ani 0.95 \
#     --out-qcov 0.85

# csvtk cut \
#     cf_v_uhvdb_votu.vclust_votu_ani.tsv \
#     --tabs \
#     --out-tabs \
#     --delete-header \
#     --fields query,reference,gani \
#     --out-file cf_v_uhvdb_votu.vclust_votu_gani.tsv.gz

# new2new + new2old + old2old
# cat combined_hq_cf_viruses.vclust_votu_gani.tsv.gz \ # new2new
#     cf_v_uhvdb_votu.vclust_votu_gani.tsv.gz \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_09/uhvdb_votu_graph.tsv.gz \ 
#     > cf_v_uhvdb_votu.vclust_votu_gani.mcl.tsv.gz

# gunzip cf_v_uhvdb_votu.vclust_votu_gani.mcl.tsv.gz

# mcl \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_votu_graph.tsv \
#     --abc \
#     -sort revsize \
#     -te 48 \
#     -o cf_v_uhvdb_votu.mcl

In [ ]:
%%bash
# !zgrep '^>' ../uhvdb_final_files/r2025_10/uhvdb_genomovars_reps.fna.gz \
#     | sed 's/^>//; s/\s.*//' > r2025_10_confident_genomovars_ids.txt

In [2]:
# new uhgv votu selection script
import polars as pl

def load_mcl_clusters(mcl, unique):
    # assign sequences to mcl clusters
    clusters = {}

    cluster_id = 0
    with open(mcl, 'r') as mcl_file:
        for line in mcl_file:
            cluster_id += 1
            for node in line.strip().split():
                clusters[node] = cluster_id
    print("Number of clusters from MCL:", cluster_id)

    # assign unclustered sequences to their own cluster
    with open(unique, 'r') as unique_file:
        for line in unique_file:
            sequence = line.strip().split()[0]
            if sequence not in clusters:
                cluster_id += 1
                clusters[sequence] = cluster_id

    print("Number of clusters:", cluster_id)

    return clusters

def load_metadata(mine_report, uhgv_metadata, clusters):
    mine_report = (
        # load mine report and join with uhvdb metadata
        pl.read_csv(mine_report, separator='\t', columns=['seq_name', 'contig_length', 'proviral_length', 'viral_genes', 'completeness_method_2'], ignore_errors=True)
            .join(
                pl.read_csv(uhgv_metadata, separator='\t', columns=['uhgv_genome', 'genome_length', 'checkv_viral_markers', 'checkv_completeness_method'], ignore_errors=True),
                how='full', left_on='seq_name', right_on='uhgv_genome', suffix='_uhgv',
            )
            # retain only sequences that are in clusters
            .filter(
                (pl.col('seq_name').is_in(clusters.keys())) |
                (pl.col('uhgv_genome').is_in(clusters.keys()))
            )
            # create cluster_id and length columns
            .with_columns([
                pl.when(pl.col('seq_name').is_not_null())
                    .then(pl.col('seq_name'))
                    .otherwise(pl.col('uhgv_genome')).alias('contig_id'),
                pl.when(pl.col('viral_genes').is_not_null())
                    .then(pl.col('viral_genes'))
                    .otherwise(pl.col('checkv_viral_markers')).alias('viral_gene_count'),
                pl.when(pl.col('checkv_completeness_method').is_not_null())
                    .then(pl.col('checkv_completeness_method'))
                    .otherwise(pl.col('completeness_method_2')).alias('completeness_method'),
                pl.when(pl.col('contig_length').is_not_null())
                    .then(pl.col('contig_length'))
                    .when(pl.col('proviral_length').is_not_null())
                    .then(pl.col('proviral_length'))
                    .otherwise(pl.col('genome_length')).alias('length').cast(pl.Float64)
            ])
            .with_columns([pl.col('contig_id').replace_strict(clusters, default=None).alias('cluster_id')])
    )

    return mine_report

In [10]:
# vClust Cluster Reps
# 1. identify median length for each cluster
# 2. Assign singletons as vOTU reps
# 3. Assign longest DTRs (> median length) as vOTU reps
# 4. Assign linear genome with highest number of viral genes (tiebreaker: closest to expected AAI length) as vOTU reps
# 5. Output vOTU reps
# 6. Output vClust vOTU cluster information

# load cluster assignments
clusters = load_mcl_clusters('cf_v_uhvdb_votu.mcl', '../uhvdb_final_files/r2025_10/uhvdb_confident_genomovars.tsv')

# load sequence metadata
# !cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/viruses.csvtk_concat.tsv \
#     cf_viruses.csvtk_concat.tsv \
#     > r2025_10_viruses.csvtk_concat.tsv

mine_report = load_metadata(
    'r2025_10_viruses.csvtk_concat.tsv',
    '/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/uhgv_metadata.tsv',
    clusters)

Number of clusters from MCL: 49222
Number of clusters: 218732


In [11]:
# 1. calculate median length amd size each cluster
cluster_metrics = (
    mine_report.group_by('cluster_id').agg(
        [
            pl.col('length').median().alias('median_length'),
            pl.col('viral_gene_count').max().alias('max_viral_genes'),
            pl.col('contig_id').len().alias('num_seqs')
        ]
    )
)

cluster_info = (
    mine_report
        .join(cluster_metrics, on='cluster_id', how='inner')
        .filter(pl.col('cluster_id').is_not_null())
)

# 2. assign singletons as vOTU representatives
singleton_clusters = set(
    cluster_metrics.filter(pl.col('num_seqs') == 1)['cluster_id']
)
print("Number of singleton clusters:", len(singleton_clusters))

cluster_reps = (
    mine_report
        .filter(pl.col('cluster_id').is_in(singleton_clusters))['contig_id', 'cluster_id']
)

Number of singleton clusters: 155417


In [12]:
# 3. assign longest DTRs as vOTU representatives (if > median length)
dtr_cluster_reps = (
    cluster_info
        .filter(
            (
                (pl.col('completeness_method').str.contains('DTR'))
            ) &
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id']))
        )
        .filter(pl.col('genome_length') >= pl.col('median_length'))
        .sort('genome_length', descending=True)
        .group_by('cluster_id', maintain_order=True)
        .first()['contig_id', 'cluster_id']
)

print("Number of DTR cluster reps added:", dtr_cluster_reps.height)

cluster_reps = pl.concat([cluster_reps, dtr_cluster_reps])

Number of DTR cluster reps added: 5205


In [13]:
# 4. Assign linear genome closest to expected AAI length with highest number of viral genes
linear_max_viral = (
    cluster_info
        .filter(
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id'])) &
            (pl.col('viral_gene_count') == pl.col('max_viral_genes'))
        )
)

# linear_max_viral[['contig_id']].write_csv('r2025_10.vclust_votu_rep_candidates.tsv', include_header=False)

# !seqkit grep \
#     /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_genomovars_reps.fna.gz \
#     --pattern-file r2025_10.vclust_votu_rep_candidates.tsv \
#     --out-file r2025_10.vclust_votu_rep_candidates.fna

# !seqkit split2 \
#     r2025_10.vclust_votu_rep_candidates.fna \
#     --by-size 10000 \
#     --out-dir checkv_split

# !sbatch checkv.sh

In [14]:
# load checkv results
import glob

checkv_df_lst = []

for file in glob.glob('checkv.part*/completeness.tsv'):
    df = pl.read_csv(file, separator='\t', columns=['contig_id', 'aai_expected_length'], ignore_errors=True)
    checkv_df_lst.append(df)

checkv_df = pl.concat(checkv_df_lst)

# 4. Assign linear genome closest to expected AAI length with highest number of viral genes
linear_cluster_reps = (
    cluster_info
        .filter(
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id'])) &
            (pl.col('viral_gene_count') == pl.col('max_viral_genes'))
        )
        .join(checkv_df, how='left', on='contig_id')
        .with_columns([
            pl.col('aai_expected_length').cast(pl.String).str.replace('NA', pl.col('median_length')).cast(pl.Float64).alias('aai_expected_length'),
        ])
        .with_columns([
            (abs(pl.col('length').cast(pl.Float64) - pl.col('aai_expected_length').cast(pl.Float64))).alias('length_diff'),
        ])
        .sort(pl.col('length_diff'), descending=False)
        .group_by('cluster_id', maintain_order=True)
        .first()['contig_id', 'cluster_id']
)

print("Number of linear cluster reps added:", linear_cluster_reps.height)

cluster_reps = pl.concat([cluster_reps, linear_cluster_reps])

# 5. Output vOTU representatives
cluster_reps[['contig_id']].write_csv('r2025_10_uhvdb_vclust_votu_reps_final.tsv', include_header=False)

# 6. Output cluster information
(
    cluster_info
        .join(checkv_df, how='left', on='contig_id')
        [['contig_id', 'cluster_id', 'num_seqs', 'length', 'median_length', 'aai_expected_length', 'viral_gene_count', 'max_viral_genes', 'completeness_method']]
        .join(cluster_reps, on='cluster_id', how='full', suffix='_rep')
        .drop('cluster_id_rep')
        .rename({'contig_id_rep': 'votu_rep'})
        .write_csv('r2025_10_uhvdb_vclust_votu_cluster_info_final.tsv', separator='\t')
)

Number of linear cluster reps added: 58110


In [ ]:
!seqkit grep \
    ../uhvdb_final_files/r2025_10/uhvdb_genomovars_reps.fna.gz \
    --pattern-file r2025_10_uhvdb_vclust_votu_reps_final.tsv \
    --out-file /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_votu_reps.fna.gz

In [20]:
import polars as pl

# load r2025_10 uhvdb clusters
r2025_10_clusters = pl.read_csv('../uhvdb_cf_supplement/r2025_10_uhvdb_vclust_votu_cluster_info_final.tsv', separator='\t')

# load r2025_09 uhvdb clusters
r2025_09_clusters = pl.read_csv('../uhvdb_clustering/vclust/uhvdb_vclust_cluster_info_final.tsv', separator='\t')

# identify sequences in both r2025_09 and r2025_10
common_contigs = set(r2025_09_clusters['contig_id']).intersection(set(r2025_10_clusters['contig_id']))
print("Number of sequences in both r2025_09 and r2025_10:", len(common_contigs))

# filter r2025_10 contig_ids to those in common clusters
r2025_10_common = r2025_10_clusters.filter(pl.col('contig_id').is_in(common_contigs)).unique('contig_id').sort('contig_id')
r2025_09_common = r2025_09_clusters.filter(pl.col('contig_id').is_in(common_contigs)).unique('contig_id').sort('contig_id')

print("Number of sequences in r2025_10 clusters:", r2025_10_common.height)
print("Number of sequences in r2025_09 clusters:", r2025_09_common.height)

print("Number of r2025_10 clusters containing r2025_09 genomes:", r2025_10_common['cluster_id'].n_unique())
print("Number of r2025_09 clusters genomes:", r2025_09_common['cluster_id'].n_unique())

Number of sequences in both r2025_09 and r2025_10: 444232
Number of sequences in r2025_10 clusters: 444232
Number of sequences in r2025_09 clusters: 444232
Number of r2025_10 clusters containing r2025_09 genomes: 201330
Number of r2025_09 clusters genomes: 201553


In [22]:
# calculate homogeneity and completeness scores for UHGV votus
from sklearn.metrics.cluster import homogeneity_score
from sklearn.metrics.cluster import completeness_score
from sklearn.metrics import v_measure_score

# Homogeneity: measure of how often UHVDB genomes newly clustered together were also cluster together in the original UHVDB database
h_score = homogeneity_score(r2025_09_common['cluster_id'], r2025_10_common['cluster_id'])
print("UHVDB r2025_09 to r2025_10 vOTU h-score:", h_score)

# Completeness: measure of how often UHVDB genomes that were clustered together in the original UHVDB database are also clustered together in the new clustering
c_score = completeness_score(r2025_09_common['cluster_id'], r2025_10_common['cluster_id'])
print("UHVDB r2025_09 to r2025_10 vOTU c-score:", c_score)

# calculate v-measure score
v_score = v_measure_score(r2025_09_common['cluster_id'], r2025_10_common['cluster_id'])
print("UHVDB r2025_09 to r2025_10 vOTU v-score:", v_score)

UHVDB r2025_09 to r2025_10 vOTU h-score: 0.9994711904659019
UHVDB r2025_09 to r2025_10 vOTU c-score: 0.9996726112042077
UHVDB r2025_09 to r2025_10 vOTU v-score: 0.9995718906881326
